# Async

Este notebook mostra dois usos comuns de programacao assincrona com `asyncio`:

- coordenar varias tarefas que esperam ao mesmo tempo;
- fazer varias requisicoes HTTP sem bloquear o restante do codigo.

Ideias principais:

- `async def` define uma corrotina;
- `await` pausa so a corrotina atual;
- `asyncio.gather(...)` executa varias corrotinas de forma concorrente e espera todas.

In [ ]:
import asyncio

async def tarefa(nome, tempo):
    print(f'Tarefa {nome} iniciada, tempo: {tempo} segundos')
    await asyncio.sleep(tempo)
    print(f'Tarefa {nome} concluida')

async def exemplo_temporizadores():
    await asyncio.gather(
        tarefa('A', 2),
        tarefa('B', 3),
        tarefa('C', 1),
    )

def executar_corrotina(corrotina):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(corrotina)
    else:
        return loop.create_task(corrotina)

# Resultado esperado ao executar `exemplo_temporizadores()`:
# Tarefa A iniciada, tempo: 2 segundos -> a primeira corrotina e agendada.
# Tarefa B iniciada, tempo: 3 segundos -> a segunda corrotina tambem comeca sem esperar A terminar.
# Tarefa C iniciada, tempo: 1 segundos -> a terceira corrotina entra no loop de eventos.
# Tarefa C concluida -> termina primeiro porque espera menos tempo.
# Tarefa A concluida -> termina depois de 2 segundos.
# Tarefa B concluida -> termina por ultimo porque tem a maior espera.


## Primeiro exemplo

A funcao `tarefa(...)` simula uma operacao demorada usando `asyncio.sleep(...)`. Como esse `sleep` e assincrono, a espera de uma tarefa nao trava a execucao das outras.

A funcao `exemplo_temporizadores()` inicia as tres tarefas com `asyncio.gather(...)`, fazendo com que elas avancem de forma concorrente.

A funcao `executar_corrotina(...)` foi criada para funcionar bem tanto em script quanto em notebook. Em script, ela usa `asyncio.run(...)`; em notebook, ela agenda a corrotina no loop que ja estiver ativo.

In [ ]:
_ = executar_corrotina(exemplo_temporizadores())

## Resultado esperado

- As tres tarefas comecam quase ao mesmo tempo.
- `C` termina primeiro porque espera 1 segundo.
- `A` termina depois de 2 segundos.
- `B` termina por ultimo depois de 3 segundos.

Esse comportamento mostra concorrencia, nao paralelismo: as tarefas dividem o tempo do loop de eventos.

## Exemplo com requisicoes HTTP

Aqui o `async` fica mais util ainda, porque operacoes de rede passam boa parte do tempo esperando resposta.

Antes de executar esta parte, garanta que o pacote `aiohttp` esteja instalado:

```bash
pip install aiohttp
```

In [ ]:
import aiohttp

async def buscar_usuario(session, usuario_id):
    url = f'https://jsonplaceholder.typicode.com/users/{usuario_id}'
    async with session.get(url) as resposta:
        resposta.raise_for_status()
        return await resposta.json()

async def exemplo_http():
    async with aiohttp.ClientSession() as session:
        usuarios_ids = [1, 2, 3, 4, 5]
        tarefas = [buscar_usuario(session, usuario_id) for usuario_id in usuarios_ids]
        usuarios = await asyncio.gather(*tarefas)

        for usuario in usuarios:
            print(f"{usuario['id']}: {usuario['name']} - {usuario['email']}")

# Resultado esperado ao executar `exemplo_http()`:
# 1: Leanne Graham - Sincere@april.biz              -> dados do usuario de id 1.
# 2: Ervin Howell - Shanna@melissa.tv               -> dados do usuario de id 2.
# 3: Clementine Bauch - Nathan@yesenia.net          -> dados do usuario de id 3.
# 4: Patricia Lebsack - Julianne.OConner@kory.org   -> dados do usuario de id 4.
# 5: Chelsey Dietrich - Lucio_Hettinger@annie.ca    -> dados do usuario de id 5.


## Segundo exemplo

A funcao `buscar_usuario(...)` faz uma requisicao para a API `jsonplaceholder.typicode.com`, valida a resposta com `raise_for_status()` e devolve os dados em formato JSON.

Em `exemplo_http()`, a lista `tarefas` reune uma corrotina para cada usuario. Quando `asyncio.gather(*tarefas)` e executado, todas as requisicoes sao disparadas juntas, o que reduz o tempo total de espera em comparacao com uma abordagem sequencial.

Esse segundo exemplo foi separado do primeiro para evitar sobrescrever funcoes e para deixar mais claro o uso de `async` em operacoes de rede.

In [ ]:
_ = executar_corrotina(exemplo_http())